# 🏗️ Module 13 — Object-Oriented Programming, Part 1: Classes & Objects

## 🎯 What you will learn
1. Why OOP? Class vs object, the 4 pillars overview
2. Defining classes; `__init__`; `self`
3. Instance vs class attributes
4. Instance, class and static methods
5. Encapsulation: public / `_protected` / `__private` and `@property`
6. Special (dunder) methods: `__str__`, `__repr__`, `__len__`, `__eq__`, `__add__`, `__getitem__`, `__call__`…
7. Object identity, copying objects, `__slots__`
8. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐⭐ — OOP has the biggest weight in most Python papers.

---
## 1. Why OOP?
Procedural code = data + functions floating separately. **OOP bundles related data (attributes) and behaviour (methods) into one unit — an object.**

- **Class** = blueprint/template (like an architectural plan)
- **Object / instance** = a concrete thing built from the blueprint (an actual house)

The four pillars:
| Pillar | Idea | Python tool |
|---|---|---|
| **Encapsulation** | hide internals, expose a clean interface | `_x`, `__x`, `@property` |
| **Abstraction** | show *what*, hide *how* | `abc.ABC`, docstrings |
| **Inheritance** | reuse & extend a parent class | `class Child(Parent)` |
| **Polymorphism** | same interface, different behaviour | method overriding, duck typing |

Everything in Python is already an object: `type(5)`, `type("a")`, `type(len)`, even `type(type)` is `type`.

In [1]:
print(type(5), type("a"), type([1]), type(len), type(int), type(type))
print(isinstance(5, object), (5).__class__, "abc".upper.__self__)

<class 'int'> <class 'str'> <class 'list'> <class 'builtin_function_or_method'> <class 'type'> <class 'type'>
True <class 'int'> abc


## 2. Your first class

In [2]:
class Dog:
    species = "Canis familiaris"          # class attribute (shared by all dogs)

    def __init__(self, name, age):        # constructor / initialiser
        self.name = name                  # instance attributes (unique per object)
        self.age = age

    def bark(self):                       # instance method
        return f"{self.name} says Woof!"

    def birthday(self):
        self.age += 1

d1 = Dog("Rex", 3)
d2 = Dog("Bella", 5)
print(d1.name, d2.name)
print(d1.bark())
d1.birthday()
print(d1.age, d2.age)                     # independent state
print(d1.species, Dog.species, d2.species)
print(type(d1), isinstance(d1, Dog))

Rex Bella
Rex says Woof!
4 5
Canis familiaris Canis familiaris Canis familiaris
<class '__main__.Dog'> True


### Understanding `self`
- `self` is the **instance the method was called on**. Python passes it automatically: `d1.bark()` ≡ `Dog.bark(d1)`.
- It is a naming convention, not a keyword (but always use `self`).
- `__init__` is the **initialiser** (called right after the object is created by `__new__`). It returns `None`.

In [3]:
print(Dog.bark(d1))                       # exactly what d1.bark() does
print(d1.bark)                            # a "bound method"

class Empty: pass
e = Empty()
e.anything = 42                           # attributes can be added dynamically
print(e.anything, e.__dict__)
print(d1.__dict__)                        # instance attributes live in a dict

Rex says Woof!
<bound method Dog.bark of <__main__.Dog object at 0x00000263BDF0C590>>
42 {'anything': 42}
{'name': 'Rex', 'age': 4}


## 3. Instance attributes vs class attributes

In [4]:
class Counter:
    total_created = 0                     # class attribute

    def __init__(self, label):
        self.label = label                # instance attribute
        Counter.total_created += 1        # modify via CLASS name

a, b, c = Counter("a"), Counter("b"), Counter("c")
print(Counter.total_created, a.total_created)

# Gotcha: assigning through an instance CREATES an instance attribute that shadows the class one
a.total_created = 99
print(a.total_created, b.total_created, Counter.total_created)
print(a.__dict__, b.__dict__)

# Mutable class attribute = shared by everybody!
class Bad:
    items = []
    def add(self, x): self.items.append(x)
x, y = Bad(), Bad()
x.add(1)
print(y.items)          # [1]  <- shared list

class Good:
    def __init__(self): self.items = []
    def add(self, x): self.items.append(x)
p, q = Good(), Good()
p.add(1); print(q.items)

3 3
99 3 3
{'label': 'a', 'total_created': 99} {'label': 'b'}
[1]
[]


## 4. Three kinds of methods

| Kind | Decorator | First parameter | Can access |
|---|---|---|---|
| **Instance** | none | `self` | instance + class data |
| **Class** | `@classmethod` | `cls` | class data only (alternative constructors!) |
| **Static** | `@staticmethod` | nothing | neither — just a namespaced utility |

In [5]:
class Pizza:
    tax = 0.05

    def __init__(self, base, toppings):
        self.base, self.toppings = base, toppings

    def price(self):                       # instance method
        return (self.base + 10 * len(self.toppings)) * (1 + Pizza.tax)

    @classmethod
    def margherita(cls):                   # alternative constructor
        return cls(100, ["cheese"])

    @classmethod
    def from_string(cls, text):            # "150:cheese,olives"
        base, tops = text.split(":")
        return cls(int(base), tops.split(","))

    @staticmethod
    def is_valid_topping(name):            # utility, no self/cls
        return name.isalpha()

print(Pizza.margherita().price())
print(Pizza.from_string("150:cheese,olives").toppings)
print(Pizza.is_valid_topping("corn"), Pizza("x", []).is_valid_topping("c0rn"))

115.5
['cheese', 'olives']
True False


## 5. Encapsulation & access control
Python has **no true private** — it relies on conventions and name mangling.

| Syntax | Meaning |
|---|---|
| `name` | public |
| `_name` | "protected" — by convention, internal use / subclasses |
| `__name` | "private" — name-mangled to `_ClassName__name` |

In [6]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner              # public
        self._bank = "PyBank"           # protected (convention)
        self.__balance = balance        # private (mangled)

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.__balance += amount

    def get_balance(self):
        return self.__balance

acc = BankAccount("Asha", 1000)
acc.deposit(500)
print(acc.get_balance())
try:
    print(acc.__balance)
except AttributeError as e:
    print("AttributeError:", e)
print(acc._BankAccount__balance)          # name mangling: still reachable if you REALLY want
print([a for a in dir(acc) if "balance" in a])

1500
AttributeError: 'BankAccount' object has no attribute '__balance'
1500
['_BankAccount__balance', 'get_balance']


### `@property` — Pythonic getters/setters
Lets you access a method like an attribute, adding validation without changing the calling code.

In [7]:
class Circle:
    def __init__(self, radius):
        self.radius = radius              # goes through the setter!

    @property
    def radius(self):                     # getter
        return self._radius

    @radius.setter
    def radius(self, value):              # setter with validation
        if value < 0:
            raise ValueError("radius cannot be negative")
        self._radius = value

    @radius.deleter
    def radius(self):
        print("deleting radius")
        del self._radius

    @property
    def area(self):                       # read-only computed attribute
        import math
        return math.pi * self._radius ** 2

c = Circle(2)
print(c.radius, round(c.area, 2))
c.radius = 5
print(round(c.area, 2))
for bad in (lambda: setattr(c, "radius", -1), lambda: setattr(c, "area", 10)):
    try:
        bad()
    except (ValueError, AttributeError) as e:
        print(type(e).__name__, "->", e)

2 12.57
78.54
ValueError -> radius cannot be negative
AttributeError -> property 'area' of 'Circle' object has no setter


## 6. Special (dunder / magic) methods
Double-underscore methods let your objects work with Python's syntax and built-ins.

| Method | Triggered by |
|---|---|
| `__init__` | `Class(...)` |
| `__str__` | `str(obj)`, `print(obj)` (human-friendly) |
| `__repr__` | `repr(obj)`, REPL/debug (developer-friendly, ideally valid code) |
| `__len__` | `len(obj)` |
| `__getitem__ / __setitem__ / __delitem__` | `obj[i]` |
| `__contains__` | `x in obj` |
| `__iter__ / __next__` | `for x in obj` |
| `__eq__ / __lt__ / __le__ ...` | `==`, `<`, `<=` … |
| `__add__ / __sub__ / __mul__ ...` | `+`, `-`, `*` |
| `__bool__` | `bool(obj)` / `if obj:` |
| `__call__` | `obj(...)` |
| `__hash__` | `hash(obj)`, dict key / set member |
| `__enter__ / __exit__` | `with obj:` |
| `__del__` | object destroyed (unreliable) |

In [8]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __str__(self):
        return f"({self.x}, {self.y})"

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, k):                      # v * 3
        return Vector(self.x * k, self.y * k)

    def __rmul__(self, k):                     # 3 * v
        return self * k

    def __eq__(self, other):
        return isinstance(other, Vector) and (self.x, self.y) == (other.x, other.y)

    def __hash__(self):                        # needed if you define __eq__ and want hashing
        return hash((self.x, self.y))

    def __abs__(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

    def __bool__(self):
        return bool(self.x or self.y)

    def __neg__(self):
        return Vector(-self.x, -self.y)

    def __len__(self):
        return 2

    def __getitem__(self, i):
        return (self.x, self.y)[i]

    def __iter__(self):
        yield self.x
        yield self.y

v, w = Vector(3, 4), Vector(1, 2)
print(v + w, v - w, v * 2, 2 * v, -v)
print(repr(v), str(v), [v, w])
print(v == Vector(3, 4), v != w, abs(v), bool(Vector(0, 0)))
print(len(v), v[0], list(v), tuple(v))
x, y = v
print(x, y, {v: "point"}[Vector(3, 4)])
print(sorted([Vector(3, 1), Vector(1, 1)], key=abs))

(4, 6) (2, 2) (6, 8) (6, 8) (-3, -4)
Vector(3, 4) (3, 4) [Vector(3, 4), Vector(1, 2)]
True True 5.0 False
2 3 [3, 4] (3, 4)
3 4 point
[Vector(1, 1), Vector(3, 1)]


### Making a class sortable and comparable

In [9]:
from functools import total_ordering

@total_ordering                      # define __eq__ and ONE ordering; the rest are generated
class Student:
    def __init__(self, name, marks):
        self.name, self.marks = name, marks
    def __eq__(self, other): return self.marks == other.marks
    def __lt__(self, other): return self.marks < other.marks
    def __repr__(self): return f"{self.name}({self.marks})"

group = [Student("A", 70), Student("B", 95), Student("C", 82)]
print(sorted(group), max(group), Student("X", 1) <= Student("Y", 2))

[A(70), C(82), B(95)] B(95) True


### A container class: `__len__`, `__getitem__`, `__contains__`, `__iter__`

In [10]:
class Playlist:
    def __init__(self): self._songs = []
    def add(self, s): self._songs.append(s); return self     # returning self enables chaining
    def __len__(self): return len(self._songs)
    def __getitem__(self, i): return self._songs[i]
    def __contains__(self, s): return s in self._songs
    def __iter__(self): return iter(self._songs)
    def __call__(self): return f"Playlist of {len(self)} songs"

p = Playlist().add("Song A").add("Song B").add("Song C")
print(len(p), p[0], p[-1], p[1:], "Song B" in p)
for s in p: print(" ♪", s)
print(p())

3 Song A Song C ['Song B', 'Song C'] True
 ♪ Song A
 ♪ Song B
 ♪ Song C
Playlist of 3 songs


## 7. Object identity, copying, and lifecycle

In [11]:
import copy
class Box:
    def __init__(self, items): self.items = items

a = Box([1, 2])
b = a                        # same object
c = copy.copy(a)             # new Box, SAME inner list
d = copy.deepcopy(a)         # fully independent
a.items.append(3)
print(a is b, b.items, c.items, d.items)

class Temp:
    def __init__(self, n): self.n = n; print("created", n)
    def __del__(self): print("destroyed", self.n)
t = Temp(1)
del t                        # reference count -> 0 -> __del__ runs (CPython)
print("after del")

import sys
o = object()
print(sys.getrefcount(o) >= 2)

True [1, 2, 3] [1, 2, 3] [1, 2]
created 1
destroyed 1
after del
True


## 8. `__slots__` — memory-lean objects
Normal instances store attributes in a `__dict__`. `__slots__` fixes the attribute names, saving memory and preventing typos.

In [12]:
class Point:
    __slots__ = ("x", "y")
    def __init__(self, x, y): self.x, self.y = x, y

class PointDict:
    def __init__(self, x, y): self.x, self.y = x, y

import sys
p1, p2 = Point(1, 2), PointDict(1, 2)
print(sys.getsizeof(p1), sys.getsizeof(p2) + sys.getsizeof(p2.__dict__))
try:
    p1.z = 3
except AttributeError as e:
    print("AttributeError:", e)

48 344
AttributeError: 'Point' object has no attribute 'z' and no __dict__ for setting new attributes


## 9. `dataclass` — write less boilerplate
`@dataclass` auto-generates `__init__`, `__repr__`, `__eq__` (and optionally ordering, immutability).

In [13]:
from dataclasses import dataclass, field, asdict, replace

@dataclass
class Employee:
    name: str
    salary: float = 30000
    skills: list[str] = field(default_factory=list)

    def raise_salary(self, pct):
        self.salary *= 1 + pct / 100

e = Employee("Asha", skills=["py"])
e.raise_salary(10)
print(e)
print(e == Employee("Asha", 33000.0, ["py"]), asdict(e), replace(e, name="Ravi"))

@dataclass(frozen=True, order=True)      # immutable & sortable
class Version:
    major: int
    minor: int = 0
print(Version(1, 2) < Version(1, 10), hash(Version(1, 0)) == hash(Version(1, 0)))
try:
    Version(1).major = 2
except Exception as ex:
    print(type(ex).__name__)

Employee(name='Asha', salary=33000.0, skills=['py'])
True {'name': 'Asha', 'salary': 33000.0, 'skills': ['py']} Employee(name='Ravi', salary=33000.0, skills=['py'])
True True
FrozenInstanceError


## 10. A complete worked example — Library system

In [14]:
from datetime import date, timedelta

class Book:
    def __init__(self, isbn, title, author):
        self.isbn, self.title, self.author = isbn, title, author
        self.available = True
    def __str__(self):
        return f"{self.title} by {self.author} [{'in' if self.available else 'out'}]"

class Member:
    MAX_BOOKS = 2
    def __init__(self, name):
        self.name = name
        self.borrowed = {}                   # isbn -> due_date

class Library:
    FINE_PER_DAY = 2

    def __init__(self):
        self.books, self.members = {}, {}

    def add_book(self, book): self.books[book.isbn] = book
    def join(self, name):
        self.members[name] = Member(name); return self.members[name]

    def lend(self, name, isbn, today):
        m, b = self.members[name], self.books[isbn]
        if not b.available:
            raise ValueError(f"'{b.title}' is already lent out")
        if len(m.borrowed) >= Member.MAX_BOOKS:
            raise ValueError(f"{name} reached the limit of {Member.MAX_BOOKS} books")
        b.available = False
        m.borrowed[isbn] = today + timedelta(days=14)

    def give_back(self, name, isbn, today):
        m = self.members[name]
        due = m.borrowed.pop(isbn)
        self.books[isbn].available = True
        late = (today - due).days
        return max(0, late) * self.FINE_PER_DAY

lib = Library()
for i, (t, a) in enumerate([("Dune", "Herbert"), ("Emma", "Austen"), ("Ulysses", "Joyce")]):
    lib.add_book(Book(i, t, a))
lib.join("Asha")
day0 = date(2025, 1, 1)
lib.lend("Asha", 0, day0)
lib.lend("Asha", 1, day0)
for problem in (lambda: lib.lend("Asha", 2, day0), lambda: lib.lend("Asha", 0, day0)):
    try:
        problem()
    except ValueError as e:
        print("Error:", e)
print("fine:", lib.give_back("Asha", 0, day0 + timedelta(days=20)))
print(*lib.books.values(), sep="\n")

Error: Asha reached the limit of 2 books
Error: 'Dune' is already lent out
fine: 12
Dune by Herbert [in]
Emma by Austen [out]
Ulysses by Joyce [in]


---
# 🕵️ Hidden Facts & Pro Tricks

In [15]:
# 1. Classes are objects too, created by `type`. You can even build a class dynamically:
Dyn = type("Dyn", (), {"x": 10, "hello": lambda self: "hi from dynamic class"})
print(Dyn().hello(), Dyn.x)

# 2. Attribute access hooks: __getattr__ (only when missing), __setattr__, __getattribute__
class Ghost:
    def __getattr__(self, name):
        return f"<no {name}>"
print(Ghost().anything)

# 3. vars(), getattr, setattr, hasattr, delattr - reflection
class P: pass
p = P(); setattr(p, "color", "red")
print(getattr(p, "color"), hasattr(p, "size"), vars(p))

# 4. __repr__ trick: !r in f-strings and using the class name generically
class Pt:
    def __init__(self, x): self.x = x
    def __repr__(self): return f"{type(self).__name__}(x={self.x!r})"
print(Pt("a"), [Pt(1)])

# 5. Define __eq__ => __hash__ becomes None (unhashable) unless you define it too
class A:
    def __eq__(self, o): return True
try: {A()}
except TypeError as e: print(e)

# 6. Method chaining by returning self (fluent interface)
class Query:
    def __init__(self): self.parts = []
    def select(self, c): self.parts.append(f"SELECT {c}"); return self
    def where(self, w): self.parts.append(f"WHERE {w}"); return self
    def __str__(self): return " ".join(self.parts)
print(Query().select("*").where("id=1"))

# 7. Class decorators & __init_subclass__ hook to register subclasses automatically
class Plugin:
    registry = []
    def __init_subclass__(cls, **kw):
        super().__init_subclass__(**kw)
        Plugin.registry.append(cls.__name__)
class CSVPlugin(Plugin): pass
class JSONPlugin(Plugin): pass
print(Plugin.registry)

# 8. Everything is public at heart: "we are all consenting adults" (Python's philosophy)
# 9. `__class__` can be reassigned; `type(obj)` vs `obj.__class__`; `id(obj)` is unique
# 10. isinstance(obj, cls) is better than type(obj) == cls (respects inheritance)
# 11. Enums, NamedTuple, TypedDict, Protocol give lightweight alternatives to full classes
from typing import NamedTuple
class Coord(NamedTuple):
    lat: float
    lon: float
print(Coord(1.5, 2.5), Coord(1.5, 2.5).lat)

# 12. Operators on the CLASS via metaclasses (deep magic), and descriptors power @property itself
print(type(property), Circle.radius.__class__.__name__)

hi from dynamic class 10
<no anything>
red False {'color': 'red'}
Pt(x='a') [Pt(x=1)]
unhashable type: 'A'
SELECT * WHERE id=1
['CSVPlugin', 'JSONPlugin']
Coord(lat=1.5, lon=2.5) 1.5
<class 'type'> property


**Hidden fact:** `@property`, `@classmethod`, `@staticmethod`, functions-as-methods, and `__slots__` are all implemented with the **descriptor protocol** (`__get__`/`__set__`). Understanding descriptors = understanding how Python attribute lookup really works.

**Hidden fact:** method lookup order is **instance `__dict__` → class → parents (MRO)**. That's why assigning `obj.method = ...` shadows the class method for just that one object.

---
# 📝 Exam Corner

**Theory**
1. Define class, object, method, attribute, constructor. Differentiate class and object.
2. What is `self`? What is `__init__`? Can a class have multiple constructors in Python?
3. Instance vs class variables (with example).
4. Instance vs class vs static methods.
5. What is encapsulation? How is it achieved? Explain name mangling.
6. What are magic methods? Explain `__str__` vs `__repr__`.
7. What is operator overloading? Overload `+` for a `Complex` class.
8. Explain `@property`.

**Program to write:** `Student` class with name, roll, marks list; methods for average, grade, display; `__str__`; sort students by average.

**Output prediction**

In [16]:
class A:
    x = 1
    def __init__(self): self.y = 2
    def show(self): return self.x + self.y

a1, a2 = A(), A()
a1.x = 10
A.x = 100
print(a1.show(), a2.show())
print(A.x, a1.x, a2.x)
a2.z = 5
print(hasattr(a1, "z"), a2.__dict__)

class B:
    def __init__(self, v): self.v = v
    def __str__(self): return f"B<{self.v}>"
    def __repr__(self): return "B!"
print(B(1), [B(1)], str(B(2)), repr(B(2)))

12 102
100 10 100
False {'y': 2, 'z': 5}
B<1> [B!] B<2> B!


<details><summary>💡 Answers</summary>

```
12 102
100 10 100
False {'y': 2, 'z': 5}
B<1> [B!] B<2> B!
```
`a1.x = 10` created an instance attribute shadowing the class one; `A.x = 100` only affects instances without their own `x`. Lists display their items with `repr`.
</details>

---
## ✅ Summary
- Class = blueprint, object = instance; `__init__` initialises; `self` is the current instance.
- Instance attrs per object, class attrs shared; three method kinds.
- `_x` convention, `__x` mangled, `@property` for controlled access.
- Dunder methods integrate your class with Python syntax; `@dataclass` removes boilerplate.

➡️ **Next:** Module 14 — OOP Part 2: Inheritance, Polymorphism & Abstraction.